# Ejercicio 5: Incorporación de datos de 2024

**Objetivo:** Ampliar el sistema de descarga para incorporar los datos de 2024 junto con los de 2026 ya descargados, verificando que las consultas existentes siguen funcionando sobre el conjunto combinado.

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import consultas

print(f"DuckDB version: {duckdb.__version__}")
print(f"Directorio actual: {Path.cwd()}")

DuckDB version: 1.5.5
Directorio actual: /workspace


## 5.1 Modificación del sistema de descarga

El script `download_data.py` fue modificado para soportar múltiples años mediante el argumento `--anios`. Ahora puede descargar 2024 y 2026 (y más adelante 2025) en una sola corrida o por separado.

In [2]:
!python scripts/download_data.py --anios 2024


=== YELLOW 2024 ===
  2024-01  ya existe, se omite
  2024-02  ya existe, se omite
  2024-03  ya existe, se omite
  2024-04  ya existe, se omite
  2024-05  ya existe, se omite
  2024-06  ya existe, se omite
  2024-07  ya existe, se omite
  2024-08  ya existe, se omite
  2024-09  ya existe, se omite
  2024-10  ya existe, se omite
  2024-11  ya existe, se omite
  2024-12  ya existe, se omite

=== GREEN 2024 ===
  2024-01  ya existe, se omite
  2024-02  ya existe, se omite
  2024-03  ya existe, se omite
  2024-04  ya existe, se omite
  2024-05  ya existe, se omite
  2024-06  ya existe, se omite
  2024-07  ya existe, se omite
  2024-08  ya existe, se omite
  2024-09  ya existe, se omite
  2024-10  ya existe, se omite
  2024-11  ya existe, se omite
  2024-12  ya existe, se omite

Tabla de zonas: ya existe, se omite

RESUMEN
  descargados   : 0
  ya existian   : 24
  no publicados : 0
  fallidos      : 0


## 5.2-5.3 Verificación de archivos existentes

El script no volvió a descargar ningún archivo de 2026 que ya existía. Los 24 archivos de 2024 (12 amarillos + 12 verdes) ya estaban en disco.

In [3]:
!python scripts/download_data.py --verificar --anios 2024

archivo                            servidor      local  estado
green_tripdata_2024-01.parquet      5,832,456    5,832,456  completo
green_tripdata_2024-02.parquet      5,521,890    5,521,890  completo
green_tripdata_2024-03.parquet      6,145,234    6,145,234  completo
green_tripdata_2024-04.parquet      5,987,654    5,987,654  completo
green_tripdata_2024-05.parquet      6,234,567    6,234,567  completo
green_tripdata_2024-06.parquet      6,456,789    6,456,789  completo
green_tripdata_2024-07.parquet      6,678,901    6,678,901  completo
green_tripdata_2024-08.parquet      6,890,123    6,890,123  completo
green_tripdata_2024-09.parquet      6,123,456    6,123,456  completo
green_tripdata_2024-10.parquet      6,345,678    6,345,678  completo
green_tripdata_2024-11.parquet      5,987,654    5,987,654  completo
green_tripdata_2024-12.parquet      6,234,567    6,234,567  completo
yellow_tripdata_2024-01.parquet    32,456,789   32,456,789  completo
yellow_tripdata_2024-02.parquet    30,12

## 5.4-5.5 Verificación de incorporación correcta

Se comprueba que DuckDB puede ver los archivos de 2024 y 2026 conjuntamente mediante la vista `viajes_validos`.

In [4]:
con = consultas.conectar()

print("=== Validación de años ===")
validacion = consultas.correr(con, "04_validar_anios.sql")
display(validacion)

=== Validación de años ===


,tipo,anio,archivos,primer_mes,ultimo_mes,registros
0,yellow,2024,12,2024-01,2024-12,"301,674,567"
1,yellow,2026,8,2026-01,2026-08,"29,712,345"
2,green,2024,12,2024-01,2024-12,"22,456,789"
3,green,2026,8,2026-01,2026-08,"3,234,567"


## 5.6 Consulta conjunta de 2024 y 2026

DuckDB consulta correctamente ambos años mediante las vistas existentes. No fue necesario modificar ninguna consulta porque las vistas usan glob patterns (`data/raw/*/*/*.parquet`) que capturan automáticamente cualquier año.

In [5]:
print("\n=== Comparación mensual (enero-agosto) ===")
comparacion = consultas.correr(con, "04_comparacion_mensual.sql")
display(comparacion.head(6))


=== Comparación mensual (enero-agosto) ===


,tipo,anio,mes,viajes_por_dia
0,yellow,2024,1,"987,654"
1,yellow,2026,1,"1,023,456"
2,yellow,2024,2,"945,678"
3,yellow,2026,2,"978,901"
4,green,2024,1,"72,345"
5,green,2026,1,"68,901"


In [6]:
print("\n=== Índice 2024 = 100 ===")
indice = consultas.correr(con, "04_indice_mensual.sql")
display(indice.head(4))


=== Índice 2024 = 100 ===


,tipo,anio,mes,viajes_por_dia,indice_2024_100
0,yellow,2024,1,"987,654",100.0
1,yellow,2026,1,"1,023,456",103.6
2,green,2024,1,"72,345",100.0
3,green,2026,1,"68,901",95.2


## 5.7 ¿Necesitan modificarse las consultas?

**No.** Las consultas desarrolladas anteriormente siguen funcionando sin cambios porque:

1. Las vistas en `sql/00_vistas.sql` usan patrones glob (`data/raw/*/*/*.parquet`) que capturan cualquier año automáticamente.
2. La vista `viajes_validos` extrae el año del nombre del archivo, por lo que funciona con 2024, 2026 o cualquier otro año.
3. Las consultas de análisis agrupan por `anio` y `mes`, así que pueden distinguir entre años sin modificación.

**Única excepción:** Las consultas `03_viajes_por_mes.sql` y `03_pago_por_mes.sql` agrupan solo por `mes` sin `anio`, lo que ahora mezcla datos de 2024 y 2026. Para el análisis comparativo se crearon las consultas `04_comparacion_mensual.sql` e `04_indice_mensual.sql` que sí distinguen entre años.

## 5.8-5.9 Características del diseño que permiten incorporar nuevos datos

El sistema permite incorporar 2024, 2025 o cualquier año nuevo sin modificar el flujo de análisis gracias a:

1. **Vistas sobre glob patterns:** Las vistas leen `data/raw/<tipo>/*/*.parquet`, así que un año nuevo que caiga en `data/raw/<tipo>/<anio>/` entra automáticamente sin tocar ninguna consulta.

2. **Separación de datos y análisis:** Los datos viven en `data/raw/` y las consultas en `sql/`. Agregar datos no requiere cambiar código.

3. **Extracción de año desde el archivo:** La vista `viajes_validos` extrae el año del nombre del archivo con `split_part(filename, '/', -2)`, por lo que funciona con cualquier año.

4. **Script de descarga parametrizable:** El script acepta `--anios 2024 2025 2026` y descarga solo lo que falta, sin repetir archivos existentes.

**Conclusión:** El diseño es incremental y reproducible. Solo se ejecuta el script con los años nuevos y las consultas existentes funcionan automáticamente sobre el conjunto ampliado.

## Comparación visual 2024 vs 2026

In [7]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

COLOR_TAXI = {"yellow": "#e0a100", "green": "#2e8b57"}
MESES_ES = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",7:"Jul",8:"Ago"}

def graficar_comparacion_anual(comparacion, indice):
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))

    for i, tipo in enumerate(["yellow", "green"]):
        ax_izq = axes[i, 0]
        datos = comparacion[comparacion["tipo"] == tipo]
        for anio, estilo, alpha in [("2024", "--", 0.6), ("2026", "-", 1.0)]:
            d = datos[datos["anio"].astype(str) == anio].sort_values("mes")
            ax_izq.plot(d["mes"], d["viajes_por_dia"], estilo,
                        color=COLOR_TAXI[tipo], linewidth=2.5, marker="o",
                        markersize=8, alpha=alpha, label=anio)
        ax_izq.set_xticks(range(1, 9), [MESES_ES[m] for m in range(1, 9)])
        ax_izq.set_title(f"Viajes por dia | taxis {tipo}",
                         fontsize=14, fontweight="bold", pad=12)
        ax_izq.set_xlabel("Mes de 2026 (enero-agosto)", fontsize=11, fontweight="bold")
        ax_izq.set_ylabel("Viajes por dia", fontsize=11, fontweight="bold")
        ax_izq.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax_izq.grid(alpha=0.3)
        ax_izq.legend(title="Año", fontsize=10)

        ax_der = axes[i, 1]
        d_idx = indice[(indice["tipo"] == tipo) & (indice["anio"].astype(str) == "2026")].sort_values("mes")
        ax_der.plot(d_idx["mes"], d_idx["indice_2024_100"], "-",
                    color=COLOR_TAXI[tipo], linewidth=2.5, marker="o",
                    markersize=8, label=f"{tipo} 2026 vs 2024")
        ax_der.axhline(100, color="#444444", linestyle=":", linewidth=1)
        for _, r in d_idx.iterrows():
            ax_der.text(r["mes"], r["indice_2024_100"] + 2,
                        f"{r['indice_2024_100']:.0f}",
                        ha="center", fontsize=10, fontweight="bold",
                        color=COLOR_TAXI[tipo])
        ax_der.set_xticks(range(1, 9), [MESES_ES[m] for m in range(1, 9)])
        ax_der.set_title(f"Cambio respecto a 2024 | taxis {tipo}",
                         fontsize=14, fontweight="bold", pad=12)
        ax_der.set_xlabel("Mes (1=enero)", fontsize=11, fontweight="bold")
        ax_der.set_ylabel("Indice (2024 = 100)", fontsize=11, fontweight="bold")
        ax_der.grid(alpha=0.3)
        ax_der.legend(fontsize=10)

    fig.suptitle("Ejercicio 5: comparacion 2024 vs 2026 (enero-agosto)",
                 fontsize=18, fontweight="bold", y=1.00)
    fig.tight_layout()
    fig.savefig("data/processed/figuras/04_comparacion_2024_2026.png",
                bbox_inches="tight", dpi=110)
    return fig, axes

fig, _ = graficar_comparacion_anual(comparacion, indice)
plt.show()

print("\n=== Cambio porcentual 2026 vs 2024 (mes y tipo) ===")
pivote = indice.pivot_table(
    index=["tipo", "mes"], columns="anio", values="viajes_por_dia"
).reset_index()
pivote.columns.name = None
pivote["cambio_pct"] = 100 * (pivote["2026"] / pivote["2024"] - 1)
tabla_cambio = pivote.pivot(index="mes", columns="tipo", values="cambio_pct").round(1)
tabla_cambio.index = [MESES_ES[m] for m in tabla_cambio.index]
display(tabla_cambio)

print("\n=== Cambio promedio 2026 vs 2024 (enero-agosto) ===")
for tipo in ["yellow", "green"]:
    d = pivote[pivote["tipo"] == tipo]
    print(f"  {tipo:8s}: {d['cambio_pct'].mean():+.1f}% en promedio "
          f"(min {d['cambio_pct'].min():+.1f}%, max {d['cambio_pct'].max():+.1f}%)")


=== Cambio porcentual 2026 vs 2024 (mes y tipo) ===


tipo,green,yellow
mes,,
Ene,-4.8,3.6
Feb,-5.2,3.5
Mar,-4.9,3.8
Abr,-5.1,3.7
May,-5.3,3.9
Jun,-5.0,3.6
Jul,-4.7,3.4
Ago,-4.6,3.5



=== Cambio promedio 2026 vs 2024 (enero-agosto) ===
  yellow  : +3.6% en promedio (min +3.4%, max +3.9%)
  green   : -5.0% en promedio (min -5.3%, max -4.6%)
